In [147]:
from pathlib import Path
import json
from datetime import datetime

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [148]:
SILVER_FOLDER = Path("../../Data/silver")
SILVER_NLP_FOLDER = Path("../../Data/silver_nlp")
GOLD_FOLDER = Path("../../Data/gold")

GOLD_FOLDER.mkdir(parents=True, exist_ok=True)

In [149]:
# pip install transformers torch sentencepiece

In [150]:
SUMMARY_MODELS = {
    "nl": "yhavinga/mt5-base-mixednews-nl",
    "en": "facebook/bart-large-cnn",
    "default": "yhavinga/mt5-base-mixednews-nl"
}

PROMPTS = {
    "nl": (
        "Vat de onderstaande tekst feitelijk samen in het Nederlands.\n"
        "Gebruik ALLEEN informatie die letterlijk in de tekst staat.\n"
        "Verzin geen bronnen, websites, media, organisaties, personen, jaartallen of cijfers.\n"
        "Noem geen NU.nl, CBS, RIVM, ministerie, gemeente of andere instantie, tenzij die letterlijk in de tekst voorkomt.\n"
        "Schrijf niet in nieuwsartikel-stijl.\n"
        "Begin niet met zinnen zoals 'Dat blijkt uit', 'Volgens', 'NU.nl zet op een rij', of 'heeft gepubliceerd'.\n"
        "Als iets niet duidelijk in de tekst staat, laat het weg.\n\n"
        "TEKST:\n"
    ),
    "en": (
        "Summarize the text factually in English.\n"
        "Use ONLY information explicitly present in the text.\n"
        "Do not invent sources, websites, media outlets, organizations, people, dates, or numbers.\n"
        "Do not write in news-article style.\n"
        "Do not start with phrases such as 'According to', 'It was published by', or 'The report says' unless present in the text.\n"
        "If something is not clearly stated in the text, omit it.\n\n"
        "TEXT:\n"
    ),
    "default": (
        "Summarize the text factually.\n"
        "Use only information explicitly present in the text.\n"
        "Do not invent sources, organizations, people, dates, or numbers.\n\n"
        "TEXT:\n"
    )
}

In [151]:
def choose_summary_model(language):
    return SUMMARY_MODELS.get(language, SUMMARY_MODELS["default"])

In [152]:
def load_model(language):
    model_name = choose_summary_model(language)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    return model_name, tokenizer, model

In [153]:
def get_summary_length(text, ratio=0.35, min_tokens=80, max_tokens=220):
    word_count = len(text.split())
    target = int(word_count * ratio)

    return {
        "min": max(40, int(target * 0.6)),
        "max": min(max_tokens, max(min_tokens, target))
    }

In [154]:
def summarize_text(text, language, tokenizer, model, ratio=0.35, max_tokens=220):
    if not text.strip():
        return ""

    length = get_summary_length(
        text,
        ratio=ratio,
        min_tokens=80,
        max_tokens=max_tokens
    )

    prompt = PROMPTS.get(language, PROMPTS["default"]) + "\n\n" + text

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    output_ids = model.generate(
        **inputs,
        max_new_tokens=length["max"],
        min_new_tokens=length["min"],
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.5,
        do_sample=False,
        early_stopping=True
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

In [155]:
def summarize_chunks(chunks, language, tokenizer, model, max_chunks=12):
    chunk_summaries = []

    for chunk in chunks[:max_chunks]:
        text = chunk["text"]

        if len(text.split()) < 40:
            continue

        summary = summarize_text(
            text=text,
            language=language,
            tokenizer=tokenizer,
            model=model,
            ratio=0.35,
            max_tokens=180
        )

        chunk_summaries.append({
            "chunk_id": chunk["chunk_id"],
            "summary": summary
        })

    return chunk_summaries




In [156]:
def create_final_summary(chunk_summaries, language, tokenizer, model):
    combined_text = " ".join(item["summary"] for item in chunk_summaries)

    if len(combined_text.split()) < 40:
        return combined_text

    return summarize_text(
        text=combined_text,
        language=language,
        tokenizer=tokenizer,
        model=model,
        ratio=0.60,
        max_tokens=450
    )

In [157]:
def get_modeling_hints(nlp_data):
    return {
        "main_topics": nlp_data.get("modeling_hints", {}).get("main_topics", []),
        "important_entities": nlp_data.get("modeling_hints", {}).get("important_entities", [])
    }

In [158]:
def save_gold_output(document_id, language, final_summary, chunk_summaries, modeling_hints, model_name):
    output_path = GOLD_FOLDER / f"{document_id}_gold.json"

    gold_data = {
        "document_id": document_id,
        "language": language,
        "summary": final_summary,
        "chunk_summaries": chunk_summaries,
        "modeling_hints_used": modeling_hints,
        "model": {
            "task": "local_summarization",
            "name": model_name,
            "type": "local_open_source"
        },
        "processed_at": datetime.now().isoformat()
    }

    output_path.write_text(
        json.dumps(gold_data, indent=4, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"Saved Gold output: {output_path}")

In [159]:
def run_gold_layer():
    silver_files = sorted(SILVER_FOLDER.glob("doc_*_silver.json"))

    if not silver_files:
        print("No silver files found.")

    for silver_file in silver_files:
        document_id = silver_file.stem.replace("_silver", "")

        print(f"Generating summary for {document_id}...")

        silver_data = json.loads(silver_file.read_text(encoding="utf-8"))

        language = silver_data.get("language", "default")

        model_name, tokenizer, model = load_model(language)

        nlp_file = SILVER_NLP_FOLDER / f"{document_id}_nlp.json"

        if nlp_file.exists():
            nlp_data = json.loads(nlp_file.read_text(encoding="utf-8"))
            modeling_hints = get_modeling_hints(nlp_data)
        else:
            modeling_hints = {
                "main_topics": [],
                "important_entities": []
            }

        chunks = silver_data["chunks"]

        chunk_summaries = summarize_chunks(
            chunks=chunks,
            language=language,
            tokenizer=tokenizer,
            model=model
        )

        final_summary = create_final_summary(
            chunk_summaries=chunk_summaries,
            language=language,
            tokenizer=tokenizer,
            model=model
        )

        save_gold_output(
            document_id=document_id,
            language=language,
            final_summary=final_summary,
            chunk_summaries=chunk_summaries,
            modeling_hints=modeling_hints,
            model_name=model_name
        )

    print("Gold layer completed.")

In [160]:
run_gold_layer()

Generating summary for doc_01...


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=80) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main

Saved Gold output: ..\..\Data\gold\doc_01_gold.json
Gold layer completed.
